# 4. Sensitivity analysis

This notebook reproduces the report's sensitivity figures and candidate-selection robustness.

The fixed E300/D80 design is replayed under all 36 combinations of four elasticity windows, three BAL allocations, and three access-scaling values. The complete target surface is then replayed under the central specification and seven single-parameter alternatives, allowing one global candidate to be selected across every propagation time and target pair.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
for path in (SRC_DIR, SCRIPTS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = DATA_DIR / "7999"
PLOTS_DIR = PROJECT_ROOT / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

REFRESH_FROM_NETWORK = (
    os.environ.get("REFRESH_7999_SIMULATION_FROM_NETWORK", "0") == "1"
)
REUSE_OUTPUTS = os.environ.get("REUSE_7999_SIMULATION_OUTPUTS", "0") == "1"

def run_script(script: str, *args: str) -> None:
    command = [sys.executable, str(SCRIPTS_DIR / script), *map(str, args)]
    print("+", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print("project root:", PROJECT_ROOT)
print("network refresh:", REFRESH_FROM_NETWORK)
print("reuse generated outputs:", REUSE_OUTPUTS)

## Run the sensitivity simulations

This is the most expensive stage. By default it reruns both the 36-case fixed-design diagnostic and the eight complete target surfaces. Set REUSE_7999_SIMULATION_OUTPUTS=1 to regenerate tables and figures from an existing local sensitivity run.

In [ ]:
import run_slot_time_parameter_sensitivity
import make_slot_time_parameter_sensitivity

fixed_path = OUT_DIR / "slot_time_e300_d80_parameter_sensitivity.csv"
surface_path = OUT_DIR / "slot_time_parameter_surface_one_at_a_time.csv"

if not REUSE_OUTPUTS:
    run_slot_time_parameter_sensitivity.main()
elif not fixed_path.exists() or not surface_path.exists():
    raise FileNotFoundError(
        "Sensitivity outputs are missing; rerun without "
        "REUSE_7999_SIMULATION_OUTPUTS=1"
    )

make_slot_time_parameter_sensitivity.main()

## Fixed-design sensitivity figures

In [ ]:
display(Image(
    filename=PLOTS_DIR / "slot_time_substitution_parameter_sensitivity.png"
))
display(Image(
    filename=PLOTS_DIR / "slot_time_substitution_parameter_sensitivity_full_blocks.png"
))

## Reproduce the two calibration regimes

The report separates demand-feasible specifications from the 60-/75-day demand-constrained estimates. This avoids presenting uncertainty about the demand curve as if it were uncertainty about physical slot-time capacity.

In [ ]:
candidates = pd.read_csv(
    OUT_DIR / "slot_time_parameter_global_candidates.csv"
)
feasible = candidates[~candidates["window_days"].isin([60, 75])]
constrained = candidates[candidates["window_days"].isin([60, 75])]

summary = pd.DataFrame([
    {
        "calibration regime": "Demand-feasible: 21-/35-day and structural sensitivities",
        "maximum-throughput result": "4.5s, E300/D80–D90; 276–285M execution",
        "historically anchored result": "3.5–4.0s, usually E250/D52.5; 223–247M",
        "interpretation": "Propagation region is stable; exact targets vary",
    },
    {
        "calibration regime": "Demand-constrained: 60-/75-day",
        "maximum-throughput result": "Approximately 150–159M execution regardless of high target",
        "historically anchored result": "Approximately 142–146M",
        "interpretation": "Demand curve, not slot-time capacity, is binding",
    },
])
display(summary)
display(candidates)

assert len(candidates) == 8
assert set(feasible["maximum_propagation_time_s"]) == {4.5}
assert feasible["maximum_execution_target"].eq(300e6).all()
assert feasible["maximum_data_target"].min() == 80e6
assert feasible["maximum_data_target"].max() == 90e6
assert 276 <= feasible["maximum_included_execution"].min() / 1e6 < 277
assert 284 < feasible["maximum_included_execution"].max() / 1e6 <= 285
assert set(feasible["historically_anchored_propagation_time_s"]) == {3.5, 4.0}
assert 150 <= constrained["maximum_included_execution"].min() / 1e6 < 151
assert 158 < constrained["maximum_included_execution"].max() / 1e6 < 159

## Reproduced outputs

Together, the four notebooks regenerate every result table and figure cited by the dynamic report:

- the canonical multiscale empirical workload;
- the fixed-ratio target grid;
- the physical slot-time surfaces and candidate profiles;
- the 36-case fixed-design sensitivity;
- the eight complete selection-sensitivity surfaces and two-regime summary.